In [ ]:
# b. Thuật toán cắt tỉa Alpha-beta với ứng dụng vào bài toán TicTacToe

# I found this article very helpful:
# https://www.geeksforgeeks.org/minimax-algorithm-in-game-theory-set-1-introduction/

import os, math

def GetWinner(board):
    """
    Returns the winner in the current board if there is one, otherwise it returns None.
    """
    # horizontal
    if board[0] == board[1] and board[1] == board[2]:
        return board[0]
    elif board[3] == board[4] and board[4] == board[5]:
        return board[3]
    elif board[6] == board[7] and board[7] == board[8]:
        return board[6]
    # vertical
    elif board[0] == board[3] and board[3] == board[6]:
        return board[0]
    elif board[1] == board[4] and board[4] == board[7]:
        return board[1]
    elif board[2] == board[5] and board[5] == board[8]:
        return board[2]
    # diagonal
    elif board[0] == board[4] and board[4] == board[8]:
        return board[0]
    elif board[2] == board[4] and board[4] == board[6]:
        return board[2]

def PrintBoard(board):
    """
    Clears the console and prints the current board.
    """
    os.system('cls' if os.name=='nt' else 'clear')
    print(f'''
    {board[0]}|{board[1]}|{board[2]}
    {board[3]}|{board[4]}|{board[5]}
    {board[6]}|{board[7]}|{board[8]}
    ''')

def GetAvailableCells(board):
    """
    Returns a list of indices containing all available cells in a board.
    """
    available = list()
    for cell in board:
        if cell != "X" and cell != "O":
            available.append(cell)
    return available

def minimax(position, depth, alpha, beta, isMaximizing):
    """
    The AI algorithm responsible for choosing the best move. Returns best value of a move.
    """
    # evaluate current board: if maximizing player won -> return 10
    #                         if minimizing player won -> return -10
    #                         if no one is winning (tie) -> return 0

    # NOTE: Even though the following AI plays perfectly, it might
    #       choose to make a move which will result in a slower victory
    #       or a faster loss. Lets take an example and explain it
    #       Assume that there are 2 possible ways for X to win the game from a give board state.
    #       Move A : X can win in 2 move
    #       Move B : X can win in 4 moves
    #       Our evaluation will return a value of +10 for both moves A and B. Even though the move A
    #       is better because it ensures a faster victory, our AI may choose B sometimes. To overcome
    #       this problem we subtract the depth value from the evaluated score. This means that in case
    #       of a victory it will choose a the victory which takes least number of moves and in case of
    #       a loss it will try to prolong the game and play as many moves as possible.
    winner = GetWinner(position)
    if winner != None:
        return 10 - depth if winner == "X" else -10 + depth
    if len(GetAvailableCells(position)) == 0:
        return 0

    if isMaximizing:
        maxEval = -math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "X"
            Eval = minimax(position, depth + 1, alpha, beta, False)
            maxEval = max(maxEval, Eval)
            alpha = max(alpha, Eval)
            position[cell - 1] = cell
            if beta <= alpha:
                break # prune
        return maxEval
    else:
        minEval = +math.inf
        for cell in GetAvailableCells(position):
            position[cell - 1] = "O"
            Eval = minimax(position, depth + 1, alpha, beta, True)
            minEval = min(minEval, Eval)
            beta = min(beta, Eval)
            position[cell - 1] = cell
            if beta <= alpha:
                break # prune
        return minEval

def FindBestMove(currentPosition, AI):
    """
    This will return the best possible move for the player.
    Will Traverse all cells, evaluate minimax function for all empty cells.
    And return the cell with optimal value.
    Parameters:
        currentPosition (list): The current board to find best move for.
        AI (str): The AI Player ("X" or "O").
    Returns:
        int: Index of best move for the current position.
    """
    bestVal = -math.inf if AI == "X" else +math.inf
    bestMove = -1
    for cell in GetAvailableCells(currentPosition):
        currentPosition[cell - 1] = AI
        moveVal = minimax(currentPosition, 0, -math.inf, +math.inf, False if AI == "X" else True)
        currentPosition[cell - 1] = cell
        if (AI == "X" and moveVal > bestVal):
            bestMove = cell
            bestVal = moveVal
        elif (AI == "O" and moveVal < bestVal):
            bestMove = cell
            bestVal = moveVal
    return bestMove

def main():
    player = input("Play as X or O? ").strip().upper()
    AI = "O" if player == "X" else "X"
    currentGame = [*range(1, 10)]
    # X always starts first.
    currentTurn = "X"
    counter = 0
    while True:
        if currentTurn == AI:
            # NOTE: if the AI starts first, it'll always choose index 0 so to save time you could play it.
            cell = FindBestMove(currentGame, AI)
            currentGame[cell - 1] = AI
            currentTurn = player
        elif currentTurn == player:
            PrintBoard(currentGame)
            while True:
                humanInput = int(input("Enter Number: ").strip())
                if humanInput in currentGame:
                    currentGame[humanInput - 1] = player
                    currentTurn = AI
                    break
                else:
                    PrintBoard(currentGame)
                    print("Cell Not Available.")
        if GetWinner(currentGame) != None:
            PrintBoard(currentGame)
            print(f"{GetWinner(currentGame)} WON!!!")
            break
        counter += 1
        if GetWinner(currentGame) == None and counter == 9:
            PrintBoard(currentGame)
            print("Tie.")
            break

if __name__ == "__main__":
    main()


Play as X or O? O

    X|2|3
    4|5|6
    7|8|9
    
Enter Number: 5

    X|X|3
    4|O|6
    7|8|9
    
Enter Number: 3

    X|X|O
    4|O|6
    X|8|9
    
Enter Number: 4

    X|X|O
    O|O|X
    X|8|9
    
Enter Number: 9

    X|X|O
    O|O|X
    X|X|O
    
Tie.


In [4]:
# Bài 1

import math
import copy

X = "X"
O = "O"
EMPTY = " "

def initial_board():
    return [[EMPTY]*3 for _ in range(3)]

def print_board(board):
    num = 1
    for i in range(3):
        row = []
        for j in range(3):
            row.append(str(num) if board[i][j] == EMPTY else board[i][j])
            num += 1
        print(" " + " | ".join(row))
        if i < 2:
            print("---+---+---")

def number_to_pos(n):
    n -= 1
    return (n // 3, n % 3)

def pos_to_number(i, j):
    return i * 3 + j + 1

def winner(board):
    for row in board:
        if row[0] == row[1] == row[2] != EMPTY:
            return row[0]

    for col in range(3):
        if board[0][col] == board[1][col] == board[2][col] != EMPTY:
            return board[0][col]

    if board[0][0] == board[1][1] == board[2][2] != EMPTY:
        return board[0][0]

    if board[0][2] == board[1][1] == board[2][0] != EMPTY:
        return board[0][2]

    return None

def is_terminal(board):
    if winner(board):
        return True
    return all(EMPTY not in row for row in board)

def utility(board):
    w = winner(board)
    if w == X:
        return 1
    if w == O:
        return -1
    return 0

def actions(board):
    return [(i, j) for i in range(3) for j in range(3) if board[i][j] == EMPTY]

def result(board, move, player):
    new_board = copy.deepcopy(board)
    i, j = move
    new_board[i][j] = player
    return new_board

def alphabeta(board, alpha, beta, is_max):
    if is_terminal(board):
        return utility(board), None

    if is_max:
        best_val = -math.inf
        best_move = None

        for move in actions(board):
            val, _ = alphabeta(result(board, move, X), alpha, beta, False)

            if val > best_val:
                best_val = val
                best_move = move

            alpha = max(alpha, best_val)

            if beta <= alpha:
                break

        return best_val, best_move

    else:
        best_val = math.inf
        best_move = None

        for move in actions(board):
            val, _ = alphabeta(result(board, move, O), alpha, beta, True)

            if val < best_val:
                best_val = val
                best_move = move

            beta = min(beta, best_val)

            if beta <= alpha:
                break

        return best_val, best_move

board = initial_board()

user = input("Chọn X hoặc O: ").upper()
while user not in ["X", "O"]:
    user = input("Nhập lại (X/O): ").upper()

ai = O if user == X else X
turn = X

while True:
    print()
    print_board(board)

    if is_terminal(board):
        w = winner(board)
        print("\nWinner:" if w else "\nHòa!", w if w else "")
        break

    if turn == user:
        try:
            move = int(input("Chọn ô (1-9): "))
            if move < 1 or move > 9:
                continue

            i, j = number_to_pos(move)

            if board[i][j] != EMPTY:
                continue

            board[i][j] = user
            turn = ai

        except ValueError:
            continue

    else:
        _, move = alphabeta(board, -math.inf, math.inf, ai == X)

        if move is None:
            break

        i, j = move
        print(f"\nAI đánh vào ô {pos_to_number(i, j)}")

        board = result(board, move, ai)
        turn = user


Chọn X hoặc O: O

 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

AI đánh vào ô 1

 X | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9
Chọn ô (1-9): 5

 X | 2 | 3
---+---+---
 4 | O | 6
---+---+---
 7 | 8 | 9

AI đánh vào ô 2

 X | X | 3
---+---+---
 4 | O | 6
---+---+---
 7 | 8 | 9
Chọn ô (1-9): 3

 X | X | O
---+---+---
 4 | O | 6
---+---+---
 7 | 8 | 9

AI đánh vào ô 7

 X | X | O
---+---+---
 4 | O | 6
---+---+---
 X | 8 | 9
Chọn ô (1-9): 4

 X | X | O
---+---+---
 O | O | 6
---+---+---
 X | 8 | 9

AI đánh vào ô 6

 X | X | O
---+---+---
 O | O | X
---+---+---
 X | 8 | 9
Chọn ô (1-9): 9

 X | X | O
---+---+---
 O | O | X
---+---+---
 X | 8 | O

AI đánh vào ô 8

 X | X | O
---+---+---
 O | O | X
---+---+---
 X | X | O

Hòa! 
